In [ ]:
import tensorflow as tf
import numpy as np
import math


In [ ]:
pip install xlns

In [ ]:
import xlns as xl

In [ ]:
def lns_to_float(x_data):
    x = x_data // 2 #Getting the log magnitude
    s = x_data % 2 #the sign bit

    x_float = tf.pow(2.0, tf.cast(x, tf.float32) / (2**23))  #opposite shifting
    x_float = tf.where(tf.equal(s, 1), -x_float, x_float)

    return x_float
def sbdb_ufunc_ideal2(x, sign_bit):
    #print("hi", sign_bit)
    #print(((tf.math.log(1+tf.pow(2.0,tf.cast((x/2**23),tf.float32)))/tf.math.log(2.0)))*2**23) #very small so removed scaling
    #print("hello",tf.cast((((tf.math.log(tf.abs(1-2*tf.cast(sign_bit,tf.float32) +tf.pow(2.0,tf.cast((x/2**23),tf.float32))))/tf.math.log(2.0))))*2**23,tf.int64)*2)
    correction=tf.cast((((tf.math.log(tf.abs(1-2*tf.cast(sign_bit,tf.float32) +tf.pow(2.0,tf.cast((x/2**23),tf.float32))))/tf.math.log(2.0)))*2**23),tf.int64)*2 #+ tf.cast(sign_bit, tf.int64)#why adding sign_bit?? if sb==1 do substraction
    #check if any rounding etc; check source code with prints
    return correction

In [ ]:
@tf.custom_gradient
def lns_add_int64(x, y):
    #print(xl.xlnsnp(np.array(x)))
    #result = tf.add(A, B)  # Forward pass

    x=tf.convert_to_tensor(x)
    y=tf.cast(y,tf.int64)
    x=tf.cast(x,tf.int64)
    max_part = tf.maximum(x, y)
    sign_bit_x=x%2
    sign_bit_y=y%2

    #if sign_bit_x==1:
        #x=-x
    #if sign_bit_y==1:
        #y=-y

    delta = -tf.abs((x // 2) - (y // 2)) #removed sign bit
    sign_bit = (x ^ y) & 1   # 0 same sign, 1 opposite sign

    sign=0
    sign = tf.where(
    tf.logical_and(tf.equal(sign_bit, 0), tf.equal(sign_bit_x, 1)),
    tf.constant(1, dtype=tf.int64),
    tf.where(
        tf.equal(sign_bit, 1),
        tf.where(tf.greater(tf.math.floordiv(x, 2), tf.math.floordiv(y, 2)), sign_bit_x, sign_bit_y),
        tf.constant(0, dtype=tf.int64)
    )
)
    #sign=tf.where(tf.logical_and(tf.equal(sign_bit,0),tf.equal(sign_bit_x,1)),1,tf.where(tf.equal(sign_bit,1),tf.where(x//2>y//2,sign_bit_x,sign_bit_y)))

    correction = sbdb_ufunc_ideal2(delta, sign_bit)
    fsum=max_part + correction*2
    #print("hi",fsum)

    result= tf.convert_to_tensor(tf.cast(max_part + tf.cast(correction,tf.int64),tf.float32))
    #print("x=",x,"y=",y,"result=",result)

    def grad(dy,variables=None):
        # Gradients w.r.t. each input
        a=lns_to_float(x)
        b=lns_to_float(y)
        scale_a=a/(a+b)
        scale_b=b/(a+b)
        scale_a*=2**24
        scale_b*=2**24
        grad_a = dy * scale_a
        grad_b = dy * scale_b

        return grad_a, grad_b

    return result, grad
@tf.custom_gradient
def lns_sub_int64(x, y):
    result = lns_add_int64(x,-y)  # Forward pass

    def grad(dy,variables=None):
        # Gradients w.r.t. each input
        a=lns_to_float(x)
        b=lns_to_float(y)
        scale_a=a/(a-b)
        scale_b=-b/(a-b)
        scale_a*=2**24
        scale_b*=2**24
        grad_a = dy * scale_a
        grad_b = dy * scale_b


        return grad_a, grad_b

    return result, grad



In [ ]:
def lns_mul_int64(x, y):
    return x + y

def lns_div_int64(x, y):
    INT64_MIN = tf.constant(-9223372036854775808, dtype=tf.int64)  # Special zero representation in LNS
    INT64_MAX_positive = tf.constant(9223372036854775806, dtype=tf.int64) #will lead to float infinity
    INT64_MAX_negative = tf.constant(9223372036854775807, dtype=tf.int64)
    INT64_NAN = tf.constant(9223372036854775805, dtype=tf.int64)  # A placeholder for NaN


    is_nan = tf.logical_and(tf.equal(x, INT64_MIN), tf.equal(y, INT64_MIN))


    is_inf = tf.logical_and(tf.not_equal(x, INT64_MIN), tf.equal(y, INT64_MIN))


    normal_div = (x - y + (y & 1)) ^ (y & 1)
    inf_result = tf.where(x&1==1 , INT64_MAX_negative, INT64_MAX_positive)


    t_nd = tf.where(is_nan, INT64_NAN, #if else
            tf.where(is_inf, inf_result,
                     tf.where(tf.equal(x, INT64_MIN), INT64_MIN,
            normal_div)))  #otherwise


    return t_nd

In [ ]:
#these are used in the final function

@tf.custom_gradient
def lns_matmul2_int64(A, B):
    if tf.rank(A) == 0 or tf.rank(B) == 0:  # Check if A or B is a scalar
            result= lns_mul_int64(A, B)
    else:
        result = []
        for i in range(tf.shape(A)[0]):  # Rows of A
            row_result = []
            for j in range(tf.shape(B)[1]):  # Columns of B
                sum_log = None
                for k in range(tf.shape(A)[1]):  # Inner dimension
                    product_log = lns_mul_int64(A[i, k], B[k, j])  # Log-space multiplication
                    sum_log = product_log if sum_log is None else lns_add_int64(sum_log, product_log)
                row_result.append(sum_log)
            result.append(row_result)
        result= tf.convert_to_tensor(result, dtype=tf.float32)  #

    def grad(dy,variables=None):
        """Define custom gradients"""
        grad_a = lns_matmul_int64(dy, tf.transpose(B))  #grad wrt a
        grad_b = lns_matmul_int64(tf.transpose(A), dy)  # Gradient w.r.t. b
        return grad_a, grad_b

    return result, grad


In [ ]:
tf.config.run_functions_eagerly(True)

In [ ]:
from tensorflow.python.eager import execute as _execute
from tensorflow.python.framework import op_def_library as _op_def_library


def lns_matmul_int64(a, b, transpose_a=False, transpose_b=False, grad_a=False, grad_b=False, name=None):
    if tf.executing_eagerly():
        #print("eager")

        result = lns_matmul2_int64(a, b)
    else:
        #print("not eager")
        #inspired from TF library
        transpose_a = _execute.make_bool(transpose_a, "transpose_a")
        transpose_b = _execute.make_bool(transpose_b, "transpose_b")
        grad_a = _execute.make_bool(grad_a, "grad_a")
        grad_b = _execute.make_bool(grad_b, "grad_b")


        _, _, _op, _outputs = _op_def_library._apply_op_helper(
            "MatMul", a=a, b=b, transpose_a=transpose_a, transpose_b=transpose_b, #registering LNSMATMUL isn't allowed, so trying to use already registered op
            grad_a=grad_a, grad_b=grad_b, name=name
        )
        result = _outputs[0]



    return result


In [ ]:
import tensorflow as tf
import numpy as np




class LogSpaceDense(tf.keras.layers.Layer):
    def __init__(self, units, **kwargs):
        super(LogSpaceDense, self).__init__(**kwargs)
        self.units = units

    def build(self, input_shape):
        input_dim = input_shape[-1]


        self.log_w = self.add_weight(shape=(input_dim, self.units),
                                     initializer='random_normal',

                                     trainable=True,dtype=tf.float32)
        self.log_b = self.add_weight(shape=(self.units,),
                                     initializer='random_normal',

                                     trainable=True,dtype=tf.float32)




    @tf.custom_gradient
    def call(self, log_inputs):

        log_output = lns_add_int64(lns_matmul_int64(log_inputs, self.log_w), self.log_b)

        def grad(dy, variables=None):
            #print("grad",dy,variables)
            a=lns_matmul_int64(tf.transpose(log_inputs), dy)
            grad_log_w = lns_sub_int64(a, self.log_w)
            grad_log_b  = tf.reduce_sum(dy, axis=0)
            grad_inputs = lns_matmul_int64(dy, self.log_w)
            #print(grad_inputs)
            return grad_inputs, [grad_log_w, grad_log_b]

        return tf.convert_to_tensor(log_output), grad
    def compute_output_shape(self, input_shape):
        """Define the output shape explicitly"""
        return (input_shape[0], self.units)


def log_mse_loss(y_true_log, y_pred_log):
    squared_error_log=lns_mul_int64(lns_sub_int64(y_true_log, y_pred_log), lns_sub_int64(y_true_log, y_pred_log))
    log_sum_se = squared_error_log[0]
    for i in range(1, tf.shape(squared_error_log)[0]):
        log_sum_se = lns_add_int64(log_sum_se, squared_error_log[i])



    log_n = tf.math.log(tf.cast(tf.size(y_true_log), tf.float32))

    return log_sum_se-log_n  # log(MSE)

class LogSpaceSGD(tf.keras.optimizers.Optimizer):
    def __init__(self, learning_rate=0.1, name="LogSpaceSGD", **kwargs):

        super().__init__(learning_rate, **kwargs)
        self.learning_rate = tf.constant(int(np.log2(learning_rate) * (2**24)))

    def apply_gradients(self, grads_and_vars, name=None, experimental_aggregate_gradients=True):
        for grad, var in grads_and_vars:

            if grad is not None:

                #print(lns_matmul_int64(self.learning_rate, grad))
                l=lns_matmul_int64(self.learning_rate, grad)
               # print("var is",var)
                #print("matmul is",l)
                #print("grad is",grad, "lr is ", self.learning_rate)
                a=lns_sub_int64(var, l)
                #print("sub is",a)
                var.assign(a)
                #print("var after sub",var)
        return tf.no_op()


model = tf.keras.Sequential([LogSpaceDense(units=2, input_shape=(1,)), LogSpaceDense(units=1)])

model.compile(optimizer=LogSpaceSGD(learning_rate=0.1), loss=log_mse_loss)
model.summary()

x_train_fp = np.array([[1.0], [2.0], [3.0], [4.0], [5.0]], dtype=np.float32)
y_train_fp = np.array([[2.0], [4.0], [6.0], [8.0], [10.0]], dtype=np.float32)

x_train_log = tf.convert_to_tensor(xl.xlnsnp(x_train_fp).nd)
y_train_log = tf.convert_to_tensor(xl.xlnsnp(y_train_fp).nd)


model.fit(x_train_log, y_train_log, epochs=10, verbose=1)

x_test_fp = np.array([[10.0]], dtype=np.float32)
x_test_log = tf.convert_to_tensor(xl.xlnsnp(x_test_fp).nd)

prediction_log = model.predict(x_test_log)
prediction_fp = lns_to_float(prediction_log)
print(x_test_fp, prediction_fp)




<ipython-input-26-731090708272>:9: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super(LogSpaceDense, self).__init__(**kwargs)


Model: "sequential_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ log_space_dense_16 (LogSpaceDense)   │ (None, 2)                   │               4 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ log_space_dense_17 (LogSpaceDense)   │ (None, 1)                   │               3 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 7 (28.00 B)

 Trainable params: 7 (28.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10


/usr/local/lib/python3.11/dist-packages/tensorflow/python/data/ops/structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 409ms/step - loss: 131238928.0000
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - loss: 131238936.0000
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 719ms/step - loss: 131238928.0000
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - loss: 131238936.0000
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 760ms/step - loss: 131238936.0000
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 300ms/step - loss: 131238936.0000
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 219ms/step - loss: 131238936.0000
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 302ms/step - loss: 131238936.0000
Epoch 9/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 237ms/step - loss: 131238936.0000
Epoch 10/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 276ms/step - loss: 131238936.0000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
[[10.]] tf.Tensor([[23.]], shape=(1, 1), dtype=float32)


In [ ]:
"""from keras.src.ops.operation import Operation
from keras.src.ops.numpy import Matmul
from keras.src.backend import any_symbolic_tensors
def lns_matmul_int64(A, B):
    print(type(A),any_symbolic_tensors((A,B)))

    if not tf.executing_eagerly():

        #print(Matmul().symbolic_call(A, B))
        return Matmul().symbolic_call(A, B) #make graph node, but works only for keras

    return lns_matmul2_int64(A,B)


"""